# Nolan (NicheExplorer): Self-Supervised Spatial Niche Detection

This tutorial demonstrates how to use Nolan (NicheExplorer) for self-supervised identification of spatial tissue domains.

Nolan uses self-supervised learning to identify spatial niches without requiring cell type annotations. It takes precomputed embeddings (e.g., from scVI or ResolVI) and spatial coordinates to learn niche-aware representations.

## Key Features:
- Self-supervised niche learning (no labels required)
- Works with any embedding (scVI, ResolVI, PCA, etc.)
- Learns tissue architecture patterns
- Supports clustering for discrete niche assignments

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Import spatialvi
import spatialvi
from spatialvi.external import Nolan

sc.set_figure_params(figsize=(6, 6))
print("spatialvi version:", spatialvi.__version__)

## 1. Load Data

In [ ]:
# Load example spatial data
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()

print(adata)

In [ ]:
# Basic preprocessing
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3")
adata = adata[:, adata.var.highly_variable].copy()

# Normalize and log transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print(f"Preprocessed data: {adata.n_obs} spots, {adata.n_vars} genes")

## 2. Create Initial Embeddings

Nolan requires precomputed embeddings. We'll use PCA as an example, but you could also use scVI or ResolVI embeddings.

In [ ]:
# Compute PCA
sc.pp.scale(adata)
sc.tl.pca(adata, n_comps=50)

# Store as X_scVI (Nolan's default expected key)
# In practice, you would compute this using scVI:
# scvi.model.SCVI.setup_anndata(adata)
# model = scvi.model.SCVI(adata)
# model.train()
# adata.obsm["X_scVI"] = model.get_latent_representation()

adata.obsm["X_scVI"] = adata.obsm["X_pca"][:, :30]  # Use first 30 PCs

print(f"Embedding shape: {adata.obsm['X_scVI'].shape}")

In [ ]:
# Visualize the spatial data
sc.pl.spatial(adata, color="total_counts", spot_size=100)

## 3. Initialize Nolan Model

In [ ]:
# Initialize Nolan model
model = Nolan(
    adata,
    emb_key="X_scVI",  # Key for input embeddings
    spatial_key="spatial",  # Key for spatial coordinates
    batch_key=None,  # Optional: batch identifier
    num_niches=50,  # Number of niche dimensions
)

print("Nolan model initialized")
print(f"Number of niches: {model.num_niches}")

## 4. Train the Model

Note: Training requires the `nolan` package to be installed.

In [ ]:
# Train the model
# Note: This requires the nolan package to be installed
try:
    model.train(
        ckpt_dir=None,  # Directory for checkpoints
        num_epochs=50,  # Number of training epochs
    )
    print("Training complete!")
except ImportError as e:
    print(f"Note: {e}")
    print("Install with: pip install nolan")
    print("Continuing with simulated embeddings for demonstration...")

    # Create simulated NOLAN embeddings for demonstration
    np.random.seed(42)
    adata.obsm["X_nolan"] = np.random.randn(adata.n_obs, model.num_niches)

## 5. Get Niche Embeddings

In [ ]:
# Get niche embeddings
try:
    adata = model.predict(
        adata=None,  # Use training data
        batch_size=2048,
        store_key="X_nolan",  # Key to store embeddings
    )
except RuntimeError:
    print("Using pre-computed embeddings (model not trained)")

print(f"Niche embeddings shape: {adata.obsm['X_nolan'].shape}")

## 6. Cluster into Discrete Niches

In [ ]:
# Cluster cells into discrete niches
try:
    adata = model.get_niche_assignments(
        n_clusters=None,  # Use Leiden clustering if None
        resolution=0.5,  # Resolution for Leiden
        store_key="niche_cluster",
    )
except (RuntimeError, AttributeError):
    # Fallback: compute neighbors and clustering directly
    sc.pp.neighbors(adata, use_rep="X_nolan", n_neighbors=15)
    sc.tl.leiden(adata, resolution=0.5, key_added="niche_cluster")

print("\nNiche cluster distribution:")
print(adata.obs["niche_cluster"].value_counts())

## 7. Visualize Results

In [ ]:
# Compute UMAP on niche embeddings
sc.pp.neighbors(adata, use_rep="X_nolan", n_neighbors=15)
sc.tl.umap(adata)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.umap(adata, color="niche_cluster", ax=axes[0], show=False, title="Niche Clusters (UMAP)")
sc.pl.spatial(adata, color="niche_cluster", spot_size=80, ax=axes[1], show=False, title="Niche Clusters (Spatial)")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize first few niche dimensions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i in range(6):
    adata.obs[f"niche_dim_{i}"] = adata.obsm["X_nolan"][:, i]
    sc.pl.spatial(
        adata,
        color=f"niche_dim_{i}",
        spot_size=60,
        ax=axes[i],
        show=False,
        title=f"Niche Dimension {i}",
    )

plt.tight_layout()
plt.show()

## 8. Analyze Niche Characteristics

In [ ]:
# Find marker genes for each niche
sc.tl.rank_genes_groups(adata, "niche_cluster", method="wilcoxon")

# Plot top markers
sc.pl.rank_genes_groups(adata, n_genes=5, sharey=False)

In [ ]:
# Get top marker genes for each niche
markers = sc.get.rank_genes_groups_df(adata, group=None)

print("Top 3 marker genes per niche:")
for group in adata.obs["niche_cluster"].cat.categories[:5]:
    group_markers = markers[markers["group"] == group].head(3)["names"].tolist()
    print(f"  Niche {group}: {', '.join(group_markers)}")

## 9. Compare with Standard Clustering

In [ ]:
# Compare with standard Leiden clustering on gene expression
sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=15, key_added="pca_neighbors")
sc.tl.leiden(adata, resolution=0.5, key_added="expr_cluster", neighbors_key="pca_neighbors")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.spatial(adata, color="niche_cluster", spot_size=80, ax=axes[0], show=False, title="Nolan Niche Clusters")
sc.pl.spatial(adata, color="expr_cluster", spot_size=80, ax=axes[1], show=False, title="Expression-based Clusters")

plt.tight_layout()
plt.show()

In [ ]:
# Compute Adjusted Rand Index between clusterings
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(adata.obs["niche_cluster"], adata.obs["expr_cluster"])
nmi = normalized_mutual_info_score(adata.obs["niche_cluster"], adata.obs["expr_cluster"])

print(f"Adjusted Rand Index: {ari:.4f}")
print(f"Normalized Mutual Information: {nmi:.4f}")

## Summary

In this tutorial, we demonstrated:

1. How to prepare spatial data with embeddings for Nolan
2. How to initialize and train the Nolan model
3. How to extract niche-aware embeddings
4. How to cluster cells into discrete niches
5. How to visualize niche patterns spatially
6. How to identify niche-specific marker genes
7. How to compare with expression-based clustering

Nolan provides a self-supervised approach to identify spatial niches that captures tissue architecture without requiring cell type labels.